## Process Lab Result Data

In [ ]:
import os

import pandas as pd
import numpy as np
import re

from datetime import date

data_path = os.path.join('..', 'data/')
raw_data_path = os.path.join(data_path, 'raw_data')
processed_data_path = os.path.join(data_path, 'processed_data')

## Load Data

### Patients of Interest

In [ ]:
cols = ['master_person_id', 'inclusion_date', 'endpoint_date']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))[cols]

del cols

inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

### A&E Activity Data

In [ ]:
ae_activity_df = pd.read_csv(os.path.join(raw_data_path, "20251202_emergency_activity_search_results.csv"))

ae_activity_df['activity_date'] = pd.to_datetime(ae_activity_df['activity_date']).dt.date

### Intensive Care Activity Data

In [ ]:
ic_activity_df = pd.read_csv(os.path.join(raw_data_path, "20251202_intensive_care_activity_search_results.csv"))

ic_activity_df['activity_date'] = pd.to_datetime(ic_activity_df['activity_date']) #.dt.date

### Outpatient Appointments Data

In [ ]:
op_appt_df = pd.read_csv(os.path.join(raw_data_path, "20251203_outpatient_clinic_activity_search_results.csv"))

op_appt_df['activity_date'] = pd.to_datetime(op_appt_df['activity_date']) #.dt.date

### AKI Activity

In [ ]:
aki_activity_df = pd.read_csv(os.path.join(raw_data_path, "20251203_aki_activity_search_results.csv"))

aki_activity_df['condition_start_date'] = pd.to_datetime(aki_activity_df['condition_start_date']).dt.date

aki_activity_df['appointment_id'] = aki_activity_df['appointment_id'].apply(lambda x: pd.NA if np.isnan(x) else int(x))
aki_activity_df['encounter_id'] = aki_activity_df['encounter_id'].apply(lambda x: pd.NA if np.isnan(x) else int(x))

### Date Breakdown Data

In [ ]:
dates_breakdown_df = pd.read_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"))

dates_breakdown_df['inclusion_date'] = pd.to_datetime(dates_breakdown_df['inclusion_date']).dt.date
dates_breakdown_df['endpoint_date'] = pd.to_datetime(dates_breakdown_df['endpoint_date']).dt.date
dates_breakdown_df['start_date'] = pd.to_datetime(dates_breakdown_df['start_date']).dt.date
dates_breakdown_df['end_date'] = pd.to_datetime(dates_breakdown_df['end_date']).dt.date

## Refined Activity Data

### A&E Activity Refinement

In [ ]:
cols = ['master_person_id', 'activity_identifier', 'activity_date']

ae_activity_refined_df = dates_breakdown_df.merge(ae_activity_df[cols], how='right', on='master_person_id')

start_filter = (ae_activity_refined_df['activity_date'] >= ae_activity_refined_df['start_date'])
end_filter = (ae_activity_refined_df['activity_date'] <= ae_activity_refined_df['end_date'])

ae_activity_refined_df = ae_activity_refined_df[start_filter&end_filter].drop_duplicates().reset_index(drop=True)

ae_activity_refined_df['grouping'] = ae_activity_refined_df['grouping'].astype(int)

del start_filter, end_filter

print(f'Number of Refined & Relevant A&E Visits: {ae_activity_refined_df.shape[0]:,}')

In [ ]:
agg = {'activity_identifier' : 'count'}
grouping_cols = ['master_person_id', 'grouping']
rename_dict = {'activity_identifier': 'ae_activities_count'}

ae_activity_agg_df = ae_activity_refined_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols).rename(columns=rename_dict)

del agg, grouping_cols, rename_dict

ae_activity_agg_df.head()

### Intensive Care Activity Refinement

In [ ]:
cols = ['master_person_id', 'activity_identifier', 'activity_date']

ic_activity_refined_df = dates_breakdown_df.merge(ic_activity_df[cols], how='right', on='master_person_id')

start_filter = (ic_activity_refined_df['activity_date'] >= ic_activity_refined_df['start_date'])
end_filter = (ic_activity_refined_df['activity_date'] <= ic_activity_refined_df['end_date'])

ic_activity_refined_df = ic_activity_refined_df[start_filter&end_filter].drop_duplicates().reset_index(drop=True)

ic_activity_refined_df['grouping'] = ic_activity_refined_df['grouping'].astype(int)

del start_filter, end_filter

print(f'Number of Refined & Relevant ICU Visits: {ic_activity_refined_df.shape[0]:,}')

In [ ]:
agg = {'activity_identifier' : 'count'}
grouping_cols = ['master_person_id', 'grouping']
rename_dict = {'activity_identifier': 'ic_activities_count'}

ic_activity_agg_df = ic_activity_refined_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols).rename(columns=rename_dict)

del agg, grouping_cols, rename_dict

ic_activity_agg_df.head()

### Outpatient Appointment Refinement

In [ ]:
def attendance(visitoutcome):
    if visitoutcome in ['Attended', 'Arrived late - seen', 'Arrived']:
        return 'Attended'
    elif visitoutcome in ['Did Not Attend', 'Left before being seen', 'Arrived late - not seen']:
        return 'DNA'
    elif visitoutcome in ['Cancelled']:
        return 'Cancelled'
    else:
        return ''

In [ ]:
cols = ['master_person_id', 'activity_identifier', 'activity_date', 'activity_visitOutcome']

op_activity_refined_df = dates_breakdown_df.merge(op_appt_df[cols], how='right', on='master_person_id')

start_filter = (op_activity_refined_df['activity_date'] >= op_activity_refined_df['start_date'])
end_filter = (op_activity_refined_df['activity_date'] <= op_activity_refined_df['end_date'])

op_activity_refined_df = op_activity_refined_df[start_filter&end_filter].drop_duplicates().reset_index(drop=True)

op_activity_refined_df['grouping'] = op_activity_refined_df['grouping'].astype(int)
op_activity_refined_df['activity_visitOutcome'] = op_activity_refined_df['activity_visitOutcome'].apply(lambda visitoutcome: attendance(visitoutcome))

del start_filter, end_filter

print(f'Number of Refined & Relevant Outpatient Clinic Visits: {op_activity_refined_df.shape[0]:,}')

In [ ]:
op_activity_refined_df['activity_visitOutcome'].value_counts()

In [ ]:
op_activity_refined_df['activity_visitOutcome'].value_counts()

In [ ]:
op_activity_refined_df.head()

In [ ]:
agg = {'activity_identifier' : 'count'}
grouping_cols = ['master_person_id', 'grouping', 'activity_visitOutcome']
rename_dict = {'activity_identifier': 'op_activities_count'}

op_activity_agg_df = op_activity_refined_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols).rename(columns=rename_dict)

del agg, grouping_cols, rename_dict

op_activity_agg_df.head()

In [ ]:
op_activity_agg_df = op_activity_agg_df.pivot_table(values='op_activities_count',
                                                    index=['master_person_id', 'grouping'],
                                                    columns='activity_visitOutcome',
                                                    fill_value=0,
                                                    aggfunc='sum'
                                                    ).reset_index()

op_activity_agg_df.columns = ['master_person_id', 'grouping', 'op_attended_count', 'op_dna_count', 'op_cancelled_count']

op_activity_agg_df.head()

### AKI Activity Refinement

In [ ]:
cols = ['master_person_id', 'encounter_id', 'condition_start_date']

aki_activity_refined_df = dates_breakdown_df.merge(aki_activity_df[cols], how='right', on='master_person_id')

start_filter = (aki_activity_refined_df['condition_start_date'] >= aki_activity_refined_df['start_date'])
end_filter = (aki_activity_refined_df['condition_start_date'] <= aki_activity_refined_df['end_date'])

aki_activity_refined_df = aki_activity_refined_df[start_filter&end_filter].drop_duplicates().reset_index(drop=True)

aki_activity_refined_df['grouping'] = aki_activity_refined_df['grouping'].astype(int)

del start_filter, end_filter

print(f'Number of Refined & Relevant AKI Events: {aki_activity_refined_df.shape[0]:,}')

In [ ]:
agg = {'encounter_id' : 'count'}
grouping_cols = ['master_person_id', 'grouping']
rename_dict = {'encounter_id': 'aki_activities_count'}

aki_activity_agg_df = aki_activity_refined_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols).rename(columns=rename_dict)

del agg, grouping_cols, rename_dict

aki_activity_agg_df.head()

### Join Activity Breakdowns back to Main DataFrame

In [ ]:
merge_cols = ['master_person_id', 'grouping']

full_activity_df = dates_breakdown_df.merge(ae_activity_agg_df, how='left', on=merge_cols).merge(ic_activity_agg_df, how='left', on=merge_cols).merge(op_activity_agg_df, how='left', on=merge_cols).merge(aki_activity_agg_df, how='left', on=merge_cols)

for col in full_activity_df.columns[6:]:
    full_activity_df[col] = full_activity_df[col].fillna(0)
    full_activity_df[col] = full_activity_df[col].astype(int)

full_activity_df.head()

### Export Activity Data

In [ ]:
# --- Save Results ---
file_name = "20260306_processed_activity_data.csv"

full_activity_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")